# Small-Lesion Subset at R512 (FracAtlas)

Runs the four R512 architecture baselines on the 50 smallest GT-area images. The R256 comparison remains in the separate `test-subcat-small-r256-*` notebook. Both prompt scenarios are reported.


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1 - SETUP (PGA-UNet + Attention U-Net)
# ══════════════════════════════════════════════════════
import os, sys, gdown, torch

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
os.chdir(BASE)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

REPO = 'https://github.com/ThongLuc2k3/PGA_Unet2D.git'

# ── Repo: PGA_Unet2D (single clone; models/networks/attention_unet_2D.py now lives
# alongside PGA's prompt_unet_2D.py, no separate Attention U-Net repo needed) ──────
if not os.path.exists(f'{BASE}/PGA_Unet2D'):
    os.system(f'git clone -q --branch main --single-branch {REPO} {BASE}/PGA_Unet2D')
print('  ✅ PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation')

# ── Dataset ───────────────────────────────────────────────────────────
DS_ZIP = f'{BASE}/dataset_FracAtlas.zip'
if not os.path.exists(DS_ZIP):
    gdown.download('https://drive.google.com/uc?id=1sfMPFQvADmZLCPJC3xPyYDrblnFZ4kQv',
                   DS_ZIP, quiet=False)
for repo in ['PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation', 'PGA_Unet2D']:
    if not os.path.exists(f'{BASE}/{repo}/dataset_FracAtlas'):
        os.system(f'unzip -oq {DS_ZIP} -d {BASE}/{repo}/')
print('  ✅ Dataset ready for both repos')

# ── Checkpoints ───────────────────────────────────────────────────────
for repo, fn, drv_id in [
    ('PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation',  'pga_unet_center_mixed_x3_shift05_qhead_512_best.pth', '1AmrZUVjGt2Hdtig-OsE9MKooYmR9qNHq'),
    ('PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation', 'att_unet_best.pth', '1YYG5S1ZIxU1_osiYGqZFnZA1YNaiCyz_'),
    ('PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation', 'attunet_concat_binary_prompt_best.pth', '17CjOghhUfX-UIATeJusSZZyfssVThXjt'),
    ('PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation', 'attunet_crop_best.pth', '1Ny4tpIKPTTYcXkpByRu5S5Zpu6hbGPve'),
]:
    os.makedirs(f'{BASE}/{repo}/checkpoints', exist_ok=True)
    p = f'{BASE}/{repo}/checkpoints/{fn}'
    if not os.path.exists(p):
        gdown.download(f'https://drive.google.com/uc?id={drv_id}', p, quiet=False)
    print(f'  ✅ {fn}  {os.path.getsize(p)//1024} KB')

os.system('pip install -q tqdm opencv-python matplotlib scipy gdown')
print(f'\n✅ Setup DONE  |  device={"cuda" if torch.cuda.is_available() else "cpu"}')


---
## Part 1 - PGA-UNet (IMG_SIZE=512, 2 prompt modes)
---

In [ ]:
# ══════════════════════════════════════════════════════
# PART 1 - PGA-UNet Test (IMG_SIZE=512, 2 prompt modes)
# ══════════════════════════════════════════════════════
import os, csv, sys
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE     = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
PGA_ROOT = f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'

# ── Clear the module cache, then load PGA modules ──────────────────────────
for _k in list(sys.modules.keys()):
    if any(x in _k for x in ('dataset','models','prompt_unet')): del sys.modules[_k]
if PGA_ROOT not in sys.path: sys.path.insert(0, PGA_ROOT)
else: sys.path.remove(PGA_ROOT); sys.path.insert(0, PGA_ROOT)

import importlib as _importlib
if 'dataset' in sys.modules: del sys.modules['dataset']
_importlib.invalidate_caches()
from dataset import PromptSegmentationDataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512
IMG_DIR  = f'{PGA_ROOT}/dataset_FracAtlas/test/images'
JSON_DIR = f'{PGA_ROOT}/dataset_FracAtlas/test/annotations'
CKPT_PATH= f'{PGA_ROOT}/checkpoints/pga_unet_center_mixed_x3_shift05_qhead_512_best.pth'
os.makedirs(f'{PGA_ROOT}/results', exist_ok=True)

model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f'✅ PGA-UNet loaded  [{DEVICE}]')

def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or  not g.any(): return float(IMG_SIZE)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1,95),np.percentile(d2,95)))

def calc_metrics_img(prob_np, gt_np):
    pm=(prob_np>0.5).astype(np.float32); gm=(gt_np>0.5).astype(np.float32); eps=1e-6
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    hd95=calc_hd95(pm,gm)
    if gm.sum()==0 or pm.sum()==0: cbl=0.0
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp + eps) / (tp + fp + fn + eps)),
                precision=float(tp / (tp + fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

KEYS  = ['dice','iou','precision','recall','hd95','cbl']
HDRS  = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓','CBL↑']
MODES = ['center_zoom','center_shift']
all_image_records = {}
pga_csv_rows = []
pga_results  = {}   # used in the summary cell

for mode in MODES:
    ds     = PromptSegmentationDataset(IMG_DIR, JSON_DIR, img_size=IMG_SIZE, is_train=False, prompt_mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    groups = OrderedDict()
    for i, (img_t, mask_t, prompt_t) in enumerate(tqdm(loader, desc=f'[{mode}]')):
        img_name, _ = ds.all_samples[i]
        gt_np=mask_t[0,0].numpy(); prompt_np=prompt_t[0,0].numpy()
        with torch.no_grad():
            prob=torch.sigmoid(model(img_t.to(DEVICE),prompt_t.to(DEVICE)))[0,0].cpu().numpy()
        if img_name not in groups:
            groups[img_name]=dict(img=img_t[0,0].numpy(),gt_union=gt_np.copy(),
                                  prob_max=prob.copy(),prompts=[prompt_np])
        else:
            g=groups[img_name]
            np.maximum(g['gt_union'],gt_np,out=g['gt_union'])
            np.maximum(g['prob_max'],prob, out=g['prob_max'])
            g['prompts'].append(prompt_np)
    img_recs=[]
    for img_name in sorted(groups.keys()):
        g=groups[img_name]; m=calc_metrics_img(g['prob_max'],g['gt_union'])
        img_recs.append(dict(img_name=img_name,img=g['img'],gt=g['gt_union'],
                             prob=g['prob_max'],prompts=g['prompts'],
                             n_samples=len(g['prompts']),**m))
    all_image_records[mode]=img_recs
    m_avg={k:np.mean([r[k] for r in img_recs]) for k in KEYS}
    pga_results[mode]=m_avg
    n_imgs=len(img_recs); n_samp=sum(r['n_samples'] for r in img_recs)
    pga_csv_rows.append([mode]+[f'{m_avg[k]:.4f}' for k in KEYS]+[str(n_imgs),str(n_samp)])

bar='='*82
print(f'\n{bar}\n  PGA-UNet - Image-level metrics (GT union+max-merge)  IMG_SIZE={IMG_SIZE}\n{bar}')
print(f"  {'Mode':<16}"+''.join(f'{h:>8}' for h in HDRS)+f"  {'N_img':>6}  {'N_smp':>6}")
print(f"  {'-'*78}")
for row in pga_csv_rows:
    print(f"  {row[0]:<16}"+''.join(f'{row[i+1]:>8}' for i in range(len(KEYS)))+f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

with open(f'{PGA_ROOT}/results/pga_unet2d_test_results.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['mode']+KEYS+['N_img','N_samples']); w.writerows(pga_csv_rows)
print(f'\n✅ CSV: {PGA_ROOT}/results/pga_unet2d_test_results.csv')


### PGA-UNet visualization - images with >=2 GT polygons

In [ ]:
from qualitative_visualization import select_shared_stems, records_for_stems
from qualitative_visualization import export_qualitative_rows
# Balanced qualitative test stems for 512: prefer 5 images with >=2 polygons and 5 images with 1 polygon.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert 'all_image_records' in dir(), '❌ Run the PGA test cell first'
N_MULTI = 5
N_SINGLE = 5

BALANCED_TEST_STEMS_R512 = select_shared_stems(
    all_image_records['center_zoom'], N_MULTI, N_SINGLE)
recs = records_for_stems(all_image_records['center_zoom'], BALANCED_TEST_STEMS_R512, context='qualitative records')
N_SHOW=len(recs)
print(f'Balanced test stems ({N_SHOW}): {BALANCED_TEST_STEMS_R512}')

fig,axes=plt.subplots(N_SHOW,5,figsize=(20,4*N_SHOW))
if N_SHOW==1: axes=axes[np.newaxis,:]
fig.suptitle(f'PGA-UNet - {N_SHOW} balanced test images (5 multi-polygon + 5 single-polygon when available)',fontsize=13,y=1.001)
for ax,ct in zip(axes[0],['Input image','Image + prompts','Prediction (merged)','Ground truth','TP/FP/FN']): ax.set_title(ct,fontsize=10,fontweight='bold')
for count,rec in enumerate(recs):
    img_np=rec['img']*0.5+0.5; gt_np=(rec['gt']>0.5).astype(float); pred_np=(rec['prob']>0.5).astype(float)
    pm_merged=np.max(np.stack(rec['prompts'],axis=0),axis=0)
    tp=(pred_np*gt_np).sum(); fp=(pred_np*(1-gt_np)).sum(); fn=((1-pred_np)*gt_np).sum(); e=1e-6
    dice=float((2*tp+e)/(2*tp+fp+fn+e)); iou=float((tp + e) / (tp + fp + fn + e)); pre=float(tp / (tp + fp+e)); rec_=float((tp+e)/(tp+fn+e))
    row=axes[count]; bg=np.stack([img_np]*3,axis=-1)
    row[0].imshow(img_np,cmap='gray',vmin=0,vmax=1); row[0].set_ylabel(f'#{count+1} [{rec["n_samples"]}p]\nDice={dice:.3f}',fontsize=8)
    row[1].imshow(img_np,cmap='gray',vmin=0,vmax=1); row[1].imshow(np.where(pm_merged>0,pm_merged,np.nan),cmap='hot',alpha=0.35,vmin=0,vmax=1)
    row[2].imshow(bg); row[2].imshow(np.where(pred_np>0,pred_np,np.nan),cmap='Blues',alpha=0.60,vmin=0,vmax=1)
    row[3].imshow(bg); row[3].imshow(np.where(gt_np>0,gt_np,np.nan),cmap='Greens',alpha=0.60,vmin=0,vmax=1)
    ov=np.zeros((*gt_np.shape,3),dtype=float); ov[...,1]=pred_np*gt_np; ov[...,0]=pred_np*(1-gt_np); ov[...,2]=(1-pred_np)*gt_np
    row[4].imshow(bg); row[4].imshow(ov, alpha=0.75 * np.any(ov > 0, axis=-1),vmin=0,vmax=1)
    row[4].text(0.02,0.02,f'Dice={dice:.3f}\nIoU={iou:.3f}\nPrecision={pre:.3f}\nRecall={rec_:.3f}',transform=row[4].transAxes,fontsize=8,va='bottom',ha='left',bbox=dict(facecolor='black',alpha=0.45,pad=3,edgecolor='none'),color='white')
    for ax in row:
        ax.axis('off')
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False, edgecolor='#555555', linewidth=1.0, clip_on=False))
handles=[Patch(facecolor='green',label='TP'),Patch(facecolor='red',label='FP'),Patch(facecolor='blue',label='FN')]
fig.legend(handles=handles,loc='lower center',ncol=3,frameon=False)
plt.tight_layout(); plt.subplots_adjust(bottom=0.06)
export_qualitative_rows(fig, axes, recs)


### PGA-UNet visualization - images with >=2 GT polygons (center_shift)

In [ ]:
from qualitative_visualization import select_shared_stems, records_for_stems
from qualitative_visualization import export_qualitative_rows
# Use the same balanced test stems for the center_shift-mode PGA panel.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert 'all_image_records' in dir(), '❌ Run the PGA test cell first'
assert 'BALANCED_TEST_STEMS_R512' in dir(), '❌ Run the center_zoom PGA visualization cell first'
recs = records_for_stems(all_image_records['center_shift'], BALANCED_TEST_STEMS_R512, context='qualitative records')
N_SHOW=len(recs)
fig,axes=plt.subplots(N_SHOW,5,figsize=(20,4*N_SHOW))
if N_SHOW==1: axes=axes[np.newaxis,:]
fig.suptitle(f'PGA-UNet - {N_SHOW} balanced test images (center_shift, same stems as center_zoom)',fontsize=13,y=1.001)
for ax,ct in zip(axes[0],['Input image','Image + prompts','Prediction (merged)','Ground truth','TP/FP/FN']): ax.set_title(ct,fontsize=10,fontweight='bold')
for count,rec in enumerate(recs):
    img_np=rec['img']*0.5+0.5; gt_np=(rec['gt']>0.5).astype(float); pred_np=(rec['prob']>0.5).astype(float)
    pm_merged=np.max(np.stack(rec['prompts'],axis=0),axis=0)
    tp=(pred_np*gt_np).sum(); fp=(pred_np*(1-gt_np)).sum(); fn=((1-pred_np)*gt_np).sum(); e=1e-6
    dice=float((2*tp+e)/(2*tp+fp+fn+e)); iou=float((tp + e) / (tp + fp + fn + e)); pre=float(tp / (tp + fp+e)); rec_=float((tp+e)/(tp+fn+e))
    row=axes[count]; bg=np.stack([img_np]*3,axis=-1)
    row[0].imshow(img_np,cmap='gray',vmin=0,vmax=1); row[0].set_ylabel(f'#{count+1} [{rec["n_samples"]}p]\nDice={dice:.3f}',fontsize=8)
    row[1].imshow(img_np,cmap='gray',vmin=0,vmax=1); row[1].imshow(np.where(pm_merged>0,pm_merged,np.nan),cmap='hot',alpha=0.35,vmin=0,vmax=1)
    row[2].imshow(bg); row[2].imshow(np.where(pred_np>0,pred_np,np.nan),cmap='Blues',alpha=0.60,vmin=0,vmax=1)
    row[3].imshow(bg); row[3].imshow(np.where(gt_np>0,gt_np,np.nan),cmap='Greens',alpha=0.60,vmin=0,vmax=1)
    ov=np.zeros((*gt_np.shape,3),dtype=float); ov[...,1]=pred_np*gt_np; ov[...,0]=pred_np*(1-gt_np); ov[...,2]=(1-pred_np)*gt_np
    row[4].imshow(bg); row[4].imshow(ov, alpha=0.75 * np.any(ov > 0, axis=-1),vmin=0,vmax=1)
    row[4].text(0.02,0.02,f'Dice={dice:.3f}\nIoU={iou:.3f}\nPrecision={pre:.3f}\nRecall={rec_:.3f}',transform=row[4].transAxes,fontsize=8,va='bottom',ha='left',bbox=dict(facecolor='black',alpha=0.45,pad=3,edgecolor='none'),color='white')
    for ax in row:
        ax.axis('off')
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False, edgecolor='#555555', linewidth=1.0, clip_on=False))
handles=[Patch(facecolor='green',label='TP'),Patch(facecolor='red',label='FP'),Patch(facecolor='blue',label='FN')]
fig.legend(handles=handles,loc='lower center',ncol=3,frameon=False)
plt.tight_layout(); plt.subplots_adjust(bottom=0.06)
export_qualitative_rows(fig, axes, recs)


---
## Part 2 - Attention U-Net (IMG_SIZE=512, no prompt)
---

In [ ]:
from qualitative_visualization import export_qualitative_rows
# ══════════════════════════════════════════════════════
# PART 2 - Attention U-Net Test (IMG_SIZE=512)
# ══════════════════════════════════════════════════════
import os, csv, sys
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt

BASE      = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
UNET_ROOT = f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'

# ── Clear the module cache, then load Attention U-Net modules ─────────────────────────
for _k in list(sys.modules.keys()):
    if any(x in _k for x in ('dataset','models','attention_unet','unet')): del sys.modules[_k]
if UNET_ROOT not in sys.path: sys.path.insert(0, UNET_ROOT)
else: sys.path.remove(UNET_ROOT); sys.path.insert(0, UNET_ROOT)

from models.networks.attention_unet_2D import Attention_UNet_2D

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class ImageMaskDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, mask_dir, img_size=512, is_train=False):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.is_train = is_train
        image_names = sorted(f for f in os.listdir(image_dir)
                             if f.lower().endswith(('.png', '.jpg', '.jpeg')))
        mask_by_stem = {}
        for mask_name in sorted(os.listdir(mask_dir)):
            if not mask_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
            stem = os.path.splitext(mask_name)[0]
            if stem in mask_by_stem:
                raise ValueError(f'Duplicate mask stem: {stem}')
            mask_by_stem[stem] = mask_name
        missing = [name for name in image_names if os.path.splitext(name)[0] not in mask_by_stem]
        if missing:
            raise FileNotFoundError(f'Missing masks for {len(missing)} images: {missing[:5]}')
        self.samples = [(name, mask_by_stem[os.path.splitext(name)[0]]) for name in image_names]
        self.images = [image_name for image_name, _ in self.samples]
        self.masks = [mask_name for _, mask_name in self.samples]

    def __len__(self):
        return len(self.samples)

    def _resize_and_pad(self, array, interpolation, pad_value=0):
        h, w = array.shape[:2]
        scale = min(self.img_size / w, self.img_size / h)
        new_w = max(1, int(round(w * scale)))
        new_h = max(1, int(round(h * scale)))
        resized = cv2.resize(array, (new_w, new_h), interpolation=interpolation)
        padded = np.full((self.img_size, self.img_size), pad_value, dtype=resized.dtype)
        pad_left = (self.img_size - new_w) // 2
        pad_top = (self.img_size - new_h) // 2
        padded[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
        return padded

    def __getitem__(self, idx):
        image_name, mask_name = self.samples[idx]
        image = cv2.imread(os.path.join(self.image_dir, image_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(os.path.join(self.mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
        image = self._resize_and_pad(image, cv2.INTER_LINEAR, pad_value=0)
        mask = self._resize_and_pad(mask, cv2.INTER_NEAREST, pad_value=0)
        image = (image.astype(np.float32) / 255.0 - 0.5) / 0.5
        mask = (mask > 127).astype(np.float32)
        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)
        return image, mask

MODEL_PATH = f'{UNET_ROOT}/checkpoints/att_unet_best.pth'
IMG_SIZE   = 512

def calc_hd95_u(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or  not gt.any(): return float(IMG_SIZE)
    pe=pred^binary_erosion(pred); ge=gt^binary_erosion(gt)
    d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1,95),np.percentile(d2,95)))

def calc_cbl_u(pred_bin, gt_bin):
    if gt_bin.sum()==0: return None
    ys,xs=np.where(gt_bin)
    gt_diag=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+1e-6
    if pred_bin.sum()==0: return 0.0
    yp,xp=np.where(pred_bin)
    return float(np.clip(1.0-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/gt_diag,0.0,1.0))

def get_centroid(m):
    if m.sum()==0: return None,None
    ys,xs=np.where(m); return float(xs.mean()),float(ys.mean())

model=Attention_UNet_2D(in_channels=1,n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH,map_location=DEVICE,weights_only=True))
model.eval()
print(f'✅ Attention U-Net loaded  [{DEVICE}]  {os.path.getsize(MODEL_PATH)//1024} KB')

test_dataset=ImageMaskDataset(image_dir=f'{UNET_ROOT}/dataset_FracAtlas/test/images',
                            mask_dir =f'{UNET_ROOT}/dataset_FracAtlas/test/masks',
                            img_size=IMG_SIZE, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Image names in sorted order, matched to the DataLoader
img_names_sorted = [os.path.basename(p) for p in test_dataset.images]

# Use exactly the same qualitative image stems and order as PGA, concat, and crop.
assert 'BALANCED_TEST_STEMS_R512' in dir(), 'Run the Part 1 PGA visualization cell first'
SHOW_STEMS = list(BALANCED_TEST_STEMS_R512)
SHOW_NAMES = set(SHOW_STEMS)
unet_viz_records = {}
att_image_records = []
print(f'Visualization: {len(SHOW_STEMS)} shared images in PGA order: {SHOW_STEMS}')

unet_all_dice,unet_all_iou,unet_all_pre,unet_all_rec,unet_all_hd95,unet_all_cbl=[],[],[],[],[],[]
smooth=1e-5

with torch.no_grad():
    for idx,(images,masks) in enumerate(test_loader):
        images,masks=images.to(DEVICE),masks.to(DEVICE)
        preds=(torch.sigmoid(model(images))>0.5).float()
        img_np=(images[0,0].cpu().numpy()+1)/2.0
        gm=masks[0,0].cpu().numpy(); pm=preds[0,0].cpu().numpy()
        tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
        dice=(2*tp+smooth)/(2*tp+fp+fn+smooth); iou=(tp + smooth) / (tp + fp + fn + smooth)
        pre=tp / (tp + fp+smooth);          rec=(tp+smooth)/(tp+fn+smooth)
        hd=calc_hd95_u(pm.astype(bool),gm.astype(bool))
        cbl=calc_cbl_u(pm.astype(bool),gm.astype(bool))
        unet_all_dice.append(dice); unet_all_iou.append(iou)
        unet_all_pre.append(pre);   unet_all_rec.append(rec)
        unet_all_hd95.append(hd)
        if cbl is not None: unet_all_cbl.append(cbl)

        img_name = img_names_sorted[idx]
        att_image_records.append(dict(img_name=img_name, img=img_np, gt=gm, prob=pm,
            dice=float(dice), iou=float(iou), precision=float(pre), recall=float(rec),
            hd95=float(hd), cbl=float(cbl) if cbl is not None else 0.0))
        if img_name in SHOW_NAMES:
            unet_viz_records[img_name] = dict(
                img=img_np, gt=gm, pred=pm, dice=float(dice), iou=float(iou),
                precision=float(pre), recall=float(rec), hd95=float(hd),
                cbl=float(cbl) if cbl is not None else 0.0,
            )

# Shared qualitative panel: same stems, order, threshold, and TP/FP/FN colors as the prompt models.
from matplotlib.patches import Patch
missing_show_stems = [name for name in SHOW_STEMS if name not in unet_viz_records]
assert not missing_show_stems, f'Attention U-Net is missing qualitative stems: {missing_show_stems}'
_viz = [(name, unet_viz_records[name]) for name in SHOW_STEMS]
N_SHOW = len(_viz)
fig, axes = plt.subplots(N_SHOW, 4, figsize=(16, 4 * N_SHOW))
if N_SHOW == 1: axes = axes[np.newaxis, :]
fig.suptitle(f'Attention U-Net - {N_SHOW} shared test images (same stems and order as PGA-UNet)', fontsize=13, y=1.001)
for ax, title in zip(axes[0], ['Input', 'Prediction', 'Ground Truth', 'TP/FP/FN']):
    ax.set_title(title, fontsize=10, fontweight='bold')
for row_idx, (img_name, rec_v) in enumerate(_viz):
    img_np, gm, pm = rec_v['img'], rec_v['gt'], rec_v['pred']
    row = axes[row_idx]; bg = np.stack([img_np] * 3, axis=-1)
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{row_idx+1} {img_name}\nDice={rec_v["dice"]:.3f}', fontsize=8)
    row[1].imshow(bg); row[1].imshow(np.where(pm > 0, pm, np.nan), cmap='Blues', alpha=0.60, vmin=0, vmax=1)
    row[2].imshow(bg); row[2].imshow(np.where(gm > 0, gm, np.nan), cmap='Greens', alpha=0.60, vmin=0, vmax=1)
    overlay = np.zeros((*gm.shape, 3), dtype=float)
    overlay[..., 1] = pm * gm; overlay[..., 0] = pm * (1 - gm); overlay[..., 2] = (1 - pm) * gm
    row[3].imshow(bg); row[3].imshow(overlay, alpha=0.75 * np.any(overlay > 0, axis=-1), vmin=0, vmax=1)
    row[3].text(0.02, 0.02, f'Dice={rec_v["dice"]:.3f}\nIoU={rec_v["iou"]:.3f}\nPrecision={rec_v["precision"]:.3f}\nRecall={rec_v["recall"]:.3f}',
                transform=row[3].transAxes, fontsize=8, va='bottom', color='white',
                bbox=dict(facecolor='black', alpha=0.45, pad=3, edgecolor='none'))
    for ax in row:
        ax.axis('off')
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False, edgecolor='#555555', linewidth=1.0, clip_on=False))
fig.legend(handles=[Patch(facecolor='green', label='TP'), Patch(facecolor='red', label='FP'),
                    Patch(facecolor='blue', label='FN')], loc='lower center', ncol=3, frameon=False)
plt.tight_layout(); plt.subplots_adjust(bottom=0.06)
export_qualitative_rows(fig, axes, _viz)

# ── Final report ──────────────────────────────────────────────────────
att_unet_results = dict(dice=float(np.mean(unet_all_dice)),  iou=float(np.mean(unet_all_iou)),
                    precision=float(np.mean(unet_all_pre)), recall=float(np.mean(unet_all_rec)),
                    hd95=float(np.mean(unet_all_hd95)),  cbl=float(np.mean(unet_all_cbl)))
print('\n'+'='*60)
print('FINAL TEST RESULTS - U-NET 2D | ImgSize=512')
print('='*60)
for k,v in att_unet_results.items(): print(f'  {k:<12}: {v:.4f}')
print(f'  Total       : {len(unet_all_dice)} samples')
print('='*60)

os.makedirs(f'{UNET_ROOT}/results', exist_ok=True)
with open(f'{UNET_ROOT}/results/attention_att_unet_results.csv','w',newline='') as f:
    w=csv.writer(f)
    w.writerow(['model','dice','iou','precision','recall','hd95','cbl','n_samples'])
    w.writerow(['AttentionUNet']+[f'{att_unet_results[k]:.4f}' for k in ['dice','iou','precision','recall','hd95','cbl']]+[len(unet_all_dice)])
print(f'✅ CSV saved')


---

In [ ]:
# ══════════════════════════════════════════════════════
# PART 3 - Attention U-Net + Prompt Channel Test (IMG_SIZE=512, 2 prompt modes)
# ══════════════════════════════════════════════════════
import os, sys, csv
import numpy as np, cv2, json as _json, glob
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE     = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
PGA_ROOT = f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'
if PGA_ROOT not in sys.path: sys.path.insert(0, PGA_ROOT)
for _k in list(sys.modules.keys()):
    if 'attunet_concat_prompt' in _k: del sys.modules[_k]
from models.networks.attunet_concat_prompt import AttUNetConcatPrompt

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE      = 512
SCALE_FACTOR  = 3.0
SHIFT_RATIO   = 0.5
BINARY_PROMPT = True
VARIANT_NAME  = 'Attention U-Net + prompt channel'
CKPT_PATH     = f'{PGA_ROOT}/checkpoints/attunet_concat_binary_prompt_best.pth'
os.makedirs(f'{PGA_ROOT}/results', exist_ok=True)


from dataset import PromptSegmentationDataset
from torchvision.transforms import InterpolationMode

class ConcatPromptDataset(PromptSegmentationDataset):
    """Dataset adapter that uses the exact PGA prompt pipeline from dataset.py."""
    def __init__(self, split='test', mode='center_zoom'):
        image_dir = f'{PGA_ROOT}/dataset_FracAtlas/{split}/images'
        json_dir = f'{PGA_ROOT}/dataset_FracAtlas/{split}/annotations'
        super().__init__(
            image_dir=image_dir,
            json_dir=json_dir,
            img_size=IMG_SIZE,
            is_train=(split == 'train'),
            prompt_mode=mode,
            scale_factor=SCALE_FACTOR,
            shift_ratio=SHIFT_RATIO,
            mixed_shift_prob=0.8,
        )
        # Preserve the metadata interface used by the image-level merge cells.
        self.samples = [
            (os.path.join(image_dir, img_name), shape_idx)
            for img_name, shape_idx in self.all_samples
        ]
        self.prompt_interpolation = cv2.INTER_NEAREST
        self.prompt_augmentation_interpolation = InterpolationMode.NEAREST

    def create_plateau_heatmap(self, bbox, orig_h, orig_w):
        prompt = np.zeros((orig_h, orig_w), dtype=np.float32)
        x_min, y_min, x_max, y_max = bbox
        x_min, y_min = max(0, int(x_min)), max(0, int(y_min))
        x_max, y_max = min(orig_w, int(x_max)), min(orig_h, int(y_max))
        if x_max > x_min and y_max > y_min:
            prompt[y_min:y_max, x_min:x_max] = 1.0
        return prompt

    def __getitem__(self, idx):
        image, mask, prompt = super().__getitem__(idx)
        return image, prompt, mask


model = AttUNetConcatPrompt(in_channels=1, n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f'✅ Model loaded ({VARIANT_NAME})  device={DEVICE}')


def calc_hd95_c(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or not g.any(): return float(IMG_SIZE)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_metrics_img_c(prob_np, gt_np):
    pm = (prob_np > 0.5).astype(np.float32); gm = (gt_np > 0.5).astype(np.float32); eps = 1e-6
    tp = (pm * gm).sum(); fp = (pm * (1 - gm)).sum(); fn = ((1 - pm) * gm).sum()
    hd95 = calc_hd95_c(pm, gm)
    if gm.sum() == 0 or pm.sum() == 0: cbl = 0.0
    else:
        ys, xs = np.where(gm > 0.5); yp, xp = np.where(pm > 0.5)
        d = np.sqrt((ys.max() - ys.min()) ** 2 + (xs.max() - xs.min()) ** 2) + eps
        cbl = float(np.clip(1. - np.sqrt((xp.mean() - xs.mean()) ** 2 + (yp.mean() - ys.mean()) ** 2) / d, 0, 1))
    return dict(dice=float((2 * tp + eps) / (2 * tp + fp + fn + eps)), iou=float((tp + eps) / (tp + fp + fn + eps)),
                precision=float(tp / (tp + fp + eps)), recall=float((tp + eps) / (tp + fn + eps)),
                hd95=hd95, cbl=cbl)


KEYS  = ['dice', 'iou', 'precision', 'recall', 'hd95', 'cbl']
HDRS  = ['Dice↑', 'IoU↑', 'Prec↑', 'Rec↑', 'HD95↓', 'CBL↑']
MODES = ['center_zoom', 'center_shift']
concat_image_records = {}
concat_results = {}
concat_csv_rows = []

for mode in MODES:
    ds     = ConcatPromptDataset('test', mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=2)
    groups = OrderedDict()
    for i, (img_t, hm_t, gt_t) in enumerate(tqdm(loader, desc=f'[Concat {mode}]')):
        img_name  = os.path.basename(ds.samples[i][0])
        gt_np     = gt_t[0, 0].numpy()
        prompt_np = hm_t[0, 0].numpy()
        with torch.no_grad():
            prob = torch.sigmoid(model(img_t.to(DEVICE), hm_t.to(DEVICE)))[0, 0].cpu().numpy()
        if img_name not in groups:
            groups[img_name] = dict(img=img_t[0, 0].numpy(), gt_union=gt_np.copy(),
                                    prob_max=prob.copy(), prompts=[prompt_np])
        else:
            np.maximum(groups[img_name]['gt_union'], gt_np, out=groups[img_name]['gt_union'])
            np.maximum(groups[img_name]['prob_max'], prob, out=groups[img_name]['prob_max'])
            groups[img_name]['prompts'].append(prompt_np)
    img_recs = []
    for img_name in sorted(groups.keys()):
        g = groups[img_name]; m = calc_metrics_img_c(g['prob_max'], g['gt_union'])
        img_recs.append(dict(img_name=img_name, img=g['img'], gt=g['gt_union'],
                             prob=g['prob_max'], prompts=g['prompts'], n_samples=len(g['prompts']), **m))
    concat_image_records[mode] = img_recs
    m_avg = {k: np.mean([r[k] for r in img_recs]) for k in KEYS}
    concat_results[mode] = m_avg
    n_imgs = len(img_recs); n_samp = sum(r['n_samples'] for r in img_recs)
    concat_csv_rows.append([mode] + [f'{m_avg[k]:.4f}' for k in KEYS] + [str(n_imgs), str(n_samp)])

bar = '=' * 82
print(f'\n{bar}\n  {VARIANT_NAME} - Image-level metrics (GT union+max-merge)  IMG_SIZE={IMG_SIZE}\n{bar}')
print(f"  {'Mode':<16}" + ''.join(f'{h:>8}' for h in HDRS) + f"  {'N_img':>6}  {'N_smp':>6}")
print(f"  {'-'*78}")
for row in concat_csv_rows:
    print(f"  {row[0]:<16}" + ''.join(f'{row[i+1]:>8}' for i in range(len(KEYS))) + f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

with open(f'{PGA_ROOT}/results/attunet_concat_prompt_results.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['mode'] + KEYS + ['N_img', 'N_samples']); w.writerows(concat_csv_rows)
print(f'\n✅ CSV: {PGA_ROOT}/results/attunet_concat_prompt_results.csv')

### Attention U-Net + Prompt Channel visualization - balanced test images (center_zoom, same stems as Part 1)

In [ ]:
from qualitative_visualization import select_shared_stems, records_for_stems
from qualitative_visualization import export_qualitative_rows
# Part 3 visualization (center_zoom): reuses BALANCED_TEST_STEMS_R512 selected
# in Part 1, so the same 10 images are shown across every model's panel.
assert 'BALANCED_TEST_STEMS_R512' in dir(), '❌ Run the Part 1 PGA visualization cell first (it selects BALANCED_TEST_STEMS_R512)'
recs = records_for_stems(concat_image_records['center_zoom'], BALANCED_TEST_STEMS_R512, context='qualitative records')
N_SHOW = len(recs)
fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1: axes = axes[np.newaxis, :]
fig.suptitle(f'{VARIANT_NAME} - {N_SHOW} balanced test images (same stems as Part 1 PGA panel)', fontsize=12, y=1.001)
for ax, ct in zip(axes[0], ['Input image', 'Image + prompts', 'Prediction (merged)', 'Ground truth', 'TP/FP/FN']):
    ax.set_title(ct, fontsize=10, fontweight='bold')
for count, rec in enumerate(recs):
    img_np = rec['img'] * 0.5 + 0.5
    gt_np = (rec['gt'] > 0.5).astype(float)
    pred_np = (rec['prob'] > 0.5).astype(float)
    prompt_merged = np.max(np.stack(rec['prompts'], axis=0), axis=0) if len(rec['prompts']) else np.zeros_like(gt_np)
    tp = (pred_np * gt_np).sum(); fp = (pred_np * (1 - gt_np)).sum(); fn = ((1 - pred_np) * gt_np).sum(); eps = 1e-6
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec_ = (tp + eps) / (tp + fn + eps)
    row = axes[count]; bg = np.stack([img_np] * 3, axis=-1)
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1); row[0].set_ylabel(f'#{count+1} [{rec["n_samples"]}p]\nDice={dice:.3f}', fontsize=8)
    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1); row[1].imshow(np.where(prompt_merged>0,prompt_merged,np.nan), cmap='hot', alpha=0.35, vmin=0, vmax=1)
    row[2].imshow(bg); row[2].imshow(np.where(pred_np>0,pred_np,np.nan), cmap='Blues', alpha=0.60, vmin=0, vmax=1)
    row[3].imshow(bg); row[3].imshow(np.where(gt_np>0,gt_np,np.nan), cmap='Greens', alpha=0.60, vmin=0, vmax=1)
    overlay = np.zeros((*gt_np.shape, 3), dtype=float)
    overlay[..., 1] = pred_np * gt_np; overlay[..., 0] = pred_np * (1 - gt_np); overlay[..., 2] = (1 - pred_np) * gt_np
    row[4].imshow(bg); row[4].imshow(overlay, alpha=0.75 * np.any(overlay > 0, axis=-1), vmin=0, vmax=1)
    row[4].text(0.02, 0.02, f'Dice={dice:.3f}\nIoU={iou:.3f}\nPrecision={prec:.3f}\nRecall={rec_:.3f}',
                transform=row[4].transAxes, fontsize=8, va='bottom', ha='left',
                bbox=dict(facecolor='black', alpha=0.45, pad=3, edgecolor='none'), color='white')
    for ax in row:
        ax.axis('off')
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False, edgecolor='#555555', linewidth=1.0, clip_on=False))
handles = [Patch(facecolor='green', label='TP'), Patch(facecolor='red', label='FP'), Patch(facecolor='blue', label='FN')]
fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False)
plt.tight_layout(); plt.subplots_adjust(bottom=0.06)
export_qualitative_rows(fig, axes, recs)


### Attention U-Net + Prompt Channel visualization (center_shift, same stems)

In [ ]:
from qualitative_visualization import select_shared_stems, records_for_stems
from qualitative_visualization import export_qualitative_rows
# Part 3 visualization (center_shift), same stems as Part 3's center_zoom panel.
recs = records_for_stems(concat_image_records['center_shift'], BALANCED_TEST_STEMS_R512, context='qualitative records')
N_SHOW = len(recs)
fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1: axes = axes[np.newaxis, :]
fig.suptitle(f'{VARIANT_NAME} - {N_SHOW} balanced test images (center_shift, same stems as center_zoom)', fontsize=12, y=1.001)
for ax, ct in zip(axes[0], ['Input image', 'Image + prompts', 'Prediction (merged)', 'Ground truth', 'TP/FP/FN']):
    ax.set_title(ct, fontsize=10, fontweight='bold')
for count, rec in enumerate(recs):
    img_np = rec['img'] * 0.5 + 0.5
    gt_np = (rec['gt'] > 0.5).astype(float)
    pred_np = (rec['prob'] > 0.5).astype(float)
    prompt_merged = np.max(np.stack(rec['prompts'], axis=0), axis=0) if len(rec['prompts']) else np.zeros_like(gt_np)
    tp = (pred_np * gt_np).sum(); fp = (pred_np * (1 - gt_np)).sum(); fn = ((1 - pred_np) * gt_np).sum(); eps = 1e-6
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec_ = (tp + eps) / (tp + fn + eps)
    row = axes[count]; bg = np.stack([img_np] * 3, axis=-1)
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1); row[0].set_ylabel(f'#{count+1} [{rec["n_samples"]}p]\nDice={dice:.3f}', fontsize=8)
    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1); row[1].imshow(np.where(prompt_merged>0,prompt_merged,np.nan), cmap='hot', alpha=0.35, vmin=0, vmax=1)
    row[2].imshow(bg); row[2].imshow(np.where(pred_np>0,pred_np,np.nan), cmap='Blues', alpha=0.60, vmin=0, vmax=1)
    row[3].imshow(bg); row[3].imshow(np.where(gt_np>0,gt_np,np.nan), cmap='Greens', alpha=0.60, vmin=0, vmax=1)
    overlay = np.zeros((*gt_np.shape, 3), dtype=float)
    overlay[..., 1] = pred_np * gt_np; overlay[..., 0] = pred_np * (1 - gt_np); overlay[..., 2] = (1 - pred_np) * gt_np
    row[4].imshow(bg); row[4].imshow(overlay, alpha=0.75 * np.any(overlay > 0, axis=-1), vmin=0, vmax=1)
    row[4].text(0.02, 0.02, f'Dice={dice:.3f}\nIoU={iou:.3f}\nPrecision={prec:.3f}\nRecall={rec_:.3f}',
                transform=row[4].transAxes, fontsize=8, va='bottom', ha='left',
                bbox=dict(facecolor='black', alpha=0.45, pad=3, edgecolor='none'), color='white')
    for ax in row:
        ax.axis('off')
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False, edgecolor='#555555', linewidth=1.0, clip_on=False))
handles = [Patch(facecolor='green', label='TP'), Patch(facecolor='red', label='FP'), Patch(facecolor='blue', label='FN')]
fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False)
plt.tight_layout(); plt.subplots_adjust(bottom=0.06)
export_qualitative_rows(fig, axes, recs)


---

In [ ]:
# ══════════════════════════════════════════════════════
# PART 4 - Attention U-Net + Prompt Crop Test (IMG_SIZE=512, 2 prompt modes)
# Predicts on the image cropped to the prompt box (not fed a heatmap channel),
# prediction pasted back into the full-image frame for evaluation. Ported
# from crop-prompt-attunet-r512.ipynb's run_eval, loading a saved checkpoint
# here instead of the freshly-trained in-memory model that file uses.
# ══════════════════════════════════════════════════════
import os, sys, csv
import cv2
import numpy as np
import torch
from matplotlib.patches import Rectangle
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE     = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
PGA_ROOT = f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512

for _k in list(sys.modules.keys()):
    if any(x in _k for x in ('dataset', 'models', 'attention_unet', 'unet')): del sys.modules[_k]
if PGA_ROOT not in sys.path: sys.path.insert(0, PGA_ROOT)
else: sys.path.remove(PGA_ROOT); sys.path.insert(0, PGA_ROOT)

from dataset import PromptSegmentationDataset
from models.networks.attention_unet_2D import Attention_UNet_2D

TEST_IMAGE_DIR = f'{PGA_ROOT}/dataset_FracAtlas/test/images'
TEST_JSON_DIR  = f'{PGA_ROOT}/dataset_FracAtlas/test/annotations'


class CroppedPromptDataset(PromptSegmentationDataset):
    """Same class as crop-prompt-attunet-r512.ipynb's CroppedPromptDataset,
    re-defined here so this cell is self-contained.
    """

    @staticmethod
    def _bbox_to_int_bounds(bx_min, by_min, bx_max, by_max, orig_h, orig_w):
        ix_min = int(np.floor(bx_min)); iy_min = int(np.floor(by_min))
        ix_max = int(np.ceil(bx_max));  iy_max = int(np.ceil(by_max))
        ix_min = max(0, min(ix_min, orig_w - 1)); iy_min = max(0, min(iy_min, orig_h - 1))
        ix_max = max(ix_min + 1, min(ix_max, orig_w)); iy_max = max(iy_min + 1, min(iy_max, orig_h))
        return ix_min, iy_min, ix_max, iy_max

    def __getitem__(self, idx):
        img_name, shape_idx = self.all_samples[idx]
        base = os.path.splitext(img_name)[0]
        image = cv2.imread(os.path.join(self.image_dir, img_name), cv2.IMREAD_GRAYSCALE)
        orig_h, orig_w = image.shape
        import json as _json
        with open(os.path.join(self.json_dir, base + '.json'), 'r', encoding='utf-8') as f:
            data = _json.load(f)
        points = np.array(data['shapes'][shape_idx]['points'])
        mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
        cv2.fillPoly(mask, [points.astype(np.int32)], 255)
        x_min, y_min = np.min(points, axis=0); x_max, y_max = np.max(points, axis=0)
        if self.prompt_mode == 'center_zoom':
            bx_min, bx_max, by_min, by_max = self._center_zoom_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w)
        elif self.prompt_mode == 'center_shift':
            bx_min, bx_max, by_min, by_max = self._center_shift_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=idx)
        else:
            raise ValueError(f'Unknown prompt_mode: {self.prompt_mode}')
        ix_min, iy_min, ix_max, iy_max = self._bbox_to_int_bounds(bx_min, by_min, bx_max, by_max, orig_h, orig_w)
        image_crop = image[iy_min:iy_max, ix_min:ix_max]
        mask_crop = mask[iy_min:iy_max, ix_min:ix_max]
        image_crop = self._resize_and_pad(image_crop, cv2.INTER_LINEAR, pad_value=0)
        mask_crop = self._resize_and_pad(mask_crop, cv2.INTER_NEAREST, pad_value=0)
        image_crop = (image_crop.astype(np.float32) / 255.0 - 0.5) / 0.5
        mask_crop = (mask_crop > 127).astype(np.float32)
        image_t = torch.from_numpy(image_crop).unsqueeze(0)
        mask_t = torch.from_numpy(mask_crop).unsqueeze(0)
        mask_t = (mask_t > 0.5).float()
        bbox_t = torch.tensor([ix_min, iy_min, ix_max, iy_max], dtype=torch.long)
        orig_hw_t = torch.tensor([orig_h, orig_w], dtype=torch.long)
        return image_t, mask_t, bbox_t, orig_hw_t


def paste_prediction_back(pred_crop_bin, bbox, orig_h, orig_w, img_size):
    ix_min, iy_min, ix_max, iy_max = [int(v) for v in bbox]
    crop_w, crop_h = ix_max - ix_min, iy_max - iy_min
    scale_crop = min(img_size / crop_w, img_size / crop_h)
    new_w = max(1, int(round(crop_w * scale_crop))); new_h = max(1, int(round(crop_h * scale_crop)))
    pad_left, pad_top = (img_size - new_w) // 2, (img_size - new_h) // 2
    unpadded = pred_crop_bin[pad_top:pad_top + new_h, pad_left:pad_left + new_w]
    box_res = cv2.resize((unpadded * 255).astype(np.uint8), (crop_w, crop_h), interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros((orig_h, orig_w), dtype=np.uint8)
    canvas[iy_min:iy_max, ix_min:ix_max] = box_res
    scale_full = min(img_size / orig_w, img_size / orig_h)
    fw = max(1, int(round(orig_w * scale_full))); fh = max(1, int(round(orig_h * scale_full)))
    resized_full = cv2.resize(canvas, (fw, fh), interpolation=cv2.INTER_NEAREST)
    padded_full = np.zeros((img_size, img_size), dtype=np.uint8)
    pl, pt = (img_size - fw) // 2, (img_size - fh) // 2
    padded_full[pt:pt + fh, pl:pl + fw] = resized_full
    return (padded_full > 127).astype(np.float32)


def bbox_to_full_frame(bbox, orig_h, orig_w, img_size):
    ix_min, iy_min, ix_max, iy_max = [int(v) for v in bbox]
    scale_full = min(img_size / orig_w, img_size / orig_h)
    pl = (img_size - max(1, int(round(orig_w * scale_full)))) // 2
    pt = (img_size - max(1, int(round(orig_h * scale_full)))) // 2
    return (ix_min * scale_full + pl, iy_min * scale_full + pt, ix_max * scale_full + pl, iy_max * scale_full + pt)




def calc_hd95_p(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or not gt.any(): return float(IMG_SIZE)
    pe = pred ^ binary_erosion(pred); ge = gt ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))


def calc_cbl_p(pred_bin: np.ndarray, gt_bin: np.ndarray):
    if gt_bin.sum() == 0: return None
    ys, xs = np.where(gt_bin)
    gt_diag = np.sqrt((ys.max() - ys.min()) ** 2 + (xs.max() - xs.min()) ** 2) + 1e-6
    if pred_bin.sum() == 0: return 0.0
    yp, xp = np.where(pred_bin)
    d = np.sqrt((xp.mean() - xs.mean()) ** 2 + (yp.mean() - ys.mean()) ** 2)
    return float(np.clip(1.0 - d / gt_diag, 0.0, 1.0))


def dice_iou_pre_rec(pm, gm, smooth=1e-5):
    tp = (pm * gm).sum(); fp = (pm * (1 - gm)).sum(); fn = ((1 - pm) * gm).sum()
    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    pre = tp / (tp + fp + smooth)
    rec = (tp + smooth) / (tp + fn + smooth)
    return dice, iou, pre, rec


crop_model = Attention_UNet_2D(in_channels=1, n_classes=1).to(DEVICE)
crop_model.load_state_dict(torch.load(f'{PGA_ROOT}/checkpoints/attunet_crop_best.pth',
                                      map_location=DEVICE, weights_only=True))
crop_model.eval()
print(f'✅ Attention U-Net + prompt crop loaded  device={DEVICE}')


def run_eval_crop(prompt_mode):
    """Per-image-merged evaluation: every polygon's crop prediction is
    pasted back into shared full-image letterboxed space, then max-merged
    per image, exactly like crop-prompt-attunet-r512.ipynb's own run_eval.
    dice_crop is a simple mean of that image's per-polygon crop-frame Dice
    (no rigorous per-image merge exists for it, see that file's docstring).
    """
    crop_ds = CroppedPromptDataset(TEST_IMAGE_DIR, TEST_JSON_DIR, img_size=IMG_SIZE,
                                    is_train=False, prompt_mode=prompt_mode)
    full_ds = PromptSegmentationDataset(TEST_IMAGE_DIR, TEST_JSON_DIR, img_size=IMG_SIZE,
                                         is_train=False, prompt_mode=prompt_mode)
    crop_loader = DataLoader(crop_ds, batch_size=1, shuffle=False)
    full_loader = DataLoader(full_ds, batch_size=1, shuffle=False)

    groups = OrderedDict()
    with torch.no_grad():
        for i, ((img_crop, mask_crop, bbox, orig_hw), (img_full, mask_full, _)) in enumerate(
                zip(crop_loader, full_loader)):
            img_name = crop_ds.all_samples[i][0]
            out_crop = crop_model(img_crop.to(DEVICE))
            pred_crop = (torch.sigmoid(out_crop) > 0.5).float()[0, 0].cpu().numpy()
            gm_crop = mask_crop[0, 0].numpy()
            d_crop, _, _, _ = dice_iou_pre_rec(pred_crop, gm_crop)
            orig_h, orig_w = int(orig_hw[0, 0]), int(orig_hw[0, 1])
            pred_full = paste_prediction_back(pred_crop, bbox[0].tolist(), orig_h, orig_w, IMG_SIZE)
            gm_full = mask_full[0, 0].numpy()
            img_np = (img_full[0, 0].numpy() + 1) / 2.0
            box_full = bbox_to_full_frame(bbox[0].tolist(), orig_h, orig_w, IMG_SIZE)
            if img_name not in groups:
                groups[img_name] = dict(img=img_np, gt_union=gm_full.copy(), pred_union=pred_full.copy(),
                                         bboxes=[box_full], dice_crop_list=[d_crop])
            else:
                np.maximum(groups[img_name]['gt_union'], gm_full, out=groups[img_name]['gt_union'])
                np.maximum(groups[img_name]['pred_union'], pred_full, out=groups[img_name]['pred_union'])
                groups[img_name]['bboxes'].append(box_full)
                groups[img_name]['dice_crop_list'].append(d_crop)

    img_recs = []
    for img_name in sorted(groups.keys()):
        g = groups[img_name]
        d, i_, p, r = dice_iou_pre_rec(g['pred_union'], g['gt_union'])
        hd = calc_hd95_p(g['pred_union'].astype(bool), g['gt_union'].astype(bool))
        cbl = calc_cbl_p(g['pred_union'].astype(bool), g['gt_union'].astype(bool))
        img_recs.append(dict(
            img_name=img_name, img=g['img'], gt=g['gt_union'], pred=g['pred_union'], bboxes=g['bboxes'],
            dice_full=d, iou_full=i_, pre_full=p, rec_full=r, hd95_full=hd,
            cbl_full=cbl if cbl is not None else 0.0,
            dice_crop=float(np.mean(g['dice_crop_list'])), n_samples=len(g['dice_crop_list']),
        ))
    return {
        'dice_full': np.mean([rc['dice_full'] for rc in img_recs]),
        'iou_full':  np.mean([rc['iou_full'] for rc in img_recs]),
        'pre_full':  np.mean([rc['pre_full'] for rc in img_recs]),
        'rec_full':  np.mean([rc['rec_full'] for rc in img_recs]),
        'hd95_full': np.mean([rc['hd95_full'] for rc in img_recs]),
        'cbl_full':  np.mean([rc['cbl_full'] for rc in img_recs]),
        'dice_crop': np.mean([rc['dice_crop'] for rc in img_recs]),
        'n_img':     len(img_recs),
        'n_samples': sum(rc['n_samples'] for rc in img_recs),
        'img_recs':  img_recs,
    }


print('\n' + '=' * 90)
print('PART 4 - TWO-SCENARIO EVALUATION - ATTENTION U-NET + PROMPT CROP (per-image-merged)')
print('=' * 90)
crop_scenarios = {'center_zoom': 'center_zoom', 'center_shift': 'center_shift'}
crop_results = {}
for name, mode in crop_scenarios.items():
    crop_results[name] = run_eval_crop(mode)
    print(f"[{name}] {crop_results[name]['n_img']} images ({crop_results[name]['n_samples']} polygon samples), completed.")

crop_header = (f"\n{'Scenario':<12} {'DiceFull':>9} {'IoUFull':>9} {'PreFull':>9} "
               f"{'RecFull':>9} {'HD95Full':>10} {'CBLFull':>9} {'DiceCrop':>9} {'N_img':>6} {'N_smp':>6}")
print(crop_header)
print('-' * 96)
for name, r in crop_results.items():
    print(f"{name:<12} {r['dice_full']:>9.4f} {r['iou_full']:>9.4f} {r['pre_full']:>9.4f} "
          f"{r['rec_full']:>9.4f} {r['hd95_full']:>10.2f} {r['cbl_full']:>9.4f} {r['dice_crop']:>9.4f} "
          f"{r['n_img']:>6} {r['n_samples']:>6}")

os.makedirs(f'{PGA_ROOT}/results', exist_ok=True)
with open(f'{PGA_ROOT}/results/attunet_crop_prompt_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['model', 'scenario', 'dice_full', 'iou_full', 'precision_full',
                      'recall_full', 'hd95_full', 'cbl_full', 'dice_crop', 'n_img', 'n_samples'])
    for name, r in crop_results.items():
        writer.writerow(['AttUNet2D_CropPrompt', name, f"{r['dice_full']:.4f}", f"{r['iou_full']:.4f}",
                          f"{r['pre_full']:.4f}", f"{r['rec_full']:.4f}", f"{r['hd95_full']:.4f}",
                          f"{r['cbl_full']:.4f}", f"{r['dice_crop']:.4f}", r['n_img'], r['n_samples']])
print(f"\n✅ CSV: {PGA_ROOT}/results/attunet_crop_prompt_results.csv")


### Attention U-Net + Prompt Crop visualization - balanced test images (center_zoom, same stems as Part 1)

In [ ]:
from qualitative_visualization import select_shared_stems, records_for_stems
from qualitative_visualization import export_qualitative_rows
# Attention U-Net + Prompt Crop visualization (center_zoom) using the shared qualitative stems.
assert 'BALANCED_TEST_STEMS_R512' in dir(), 'Run the Part 1 PGA visualization cell first'
_all_recs = crop_results['center_zoom']['img_recs']
recs = records_for_stems(_all_recs, BALANCED_TEST_STEMS_R512, context='qualitative records')
N_SHOW = len(recs)
fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1: axes = axes[np.newaxis, :]
fig.suptitle(f'Attention U-Net + Prompt Crop - {N_SHOW} shared test images (center_zoom, same stems as PGA-UNet)', fontsize=13, y=1.001)
for ax, title in zip(axes[0], ['Input image', 'Prompt box', 'Prediction', 'Ground truth', 'TP/FP/FN']):
    ax.set_title(title, fontsize=10, fontweight='bold')
for row_idx, rec in enumerate(recs):
    img_np = rec['img']; pred_np = rec['pred']; gt_np = rec['gt']
    dice, iou, precision, recall = dice_iou_pre_rec(pred_np, gt_np)
    row = axes[row_idx]; bg = np.stack([img_np] * 3, axis=-1)
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{row_idx+1} [{rec["n_samples"]}p]\nDice={dice:.3f}', fontsize=8)
    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    for bx0, by0, bx1, by1 in rec['bboxes']:
        row[1].add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0,
                                   edgecolor='cyan', fill=False, linewidth=1.5))
    row[2].imshow(bg); row[2].imshow(np.where(pred_np > 0, pred_np, np.nan),
                                    cmap='Blues', alpha=0.60, vmin=0, vmax=1)
    row[3].imshow(bg); row[3].imshow(np.where(gt_np > 0, gt_np, np.nan),
                                    cmap='Greens', alpha=0.60, vmin=0, vmax=1)
    overlay = np.zeros((*gt_np.shape, 3), dtype=float)
    overlay[..., 1] = pred_np * gt_np
    overlay[..., 0] = pred_np * (1 - gt_np)
    overlay[..., 2] = (1 - pred_np) * gt_np
    row[4].imshow(bg); row[4].imshow(overlay, alpha=0.75 * np.any(overlay > 0, axis=-1), vmin=0, vmax=1)
    row[4].text(0.02, 0.02, f'Dice={dice:.3f}\nIoU={iou:.3f}\nPrecision={precision:.3f}\nRecall={recall:.3f}',
                transform=row[4].transAxes, fontsize=8, va='bottom', ha='left', color='white',
                bbox=dict(facecolor='black', alpha=0.45, pad=3, edgecolor='none'))
    for ax in row:
        ax.axis('off')
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False, edgecolor='#555555', linewidth=1.0, clip_on=False))
fig.legend(handles=[Patch(facecolor='green', label='TP'), Patch(facecolor='red', label='FP'),
                    Patch(facecolor='blue', label='FN')],
           loc='lower center', ncol=3, frameon=False)
plt.tight_layout(); plt.subplots_adjust(bottom=0.06)
export_qualitative_rows(fig, axes, recs)


### Attention U-Net + Prompt Crop visualization (center_shift, same stems)

In [ ]:
from qualitative_visualization import select_shared_stems, records_for_stems
from qualitative_visualization import export_qualitative_rows
# Attention U-Net + Prompt Crop visualization (center_shift) using the shared qualitative stems.
assert 'BALANCED_TEST_STEMS_R512' in dir(), 'Run the Part 1 PGA visualization cell first'
_all_recs = crop_results['center_shift']['img_recs']
recs = records_for_stems(_all_recs, BALANCED_TEST_STEMS_R512, context='qualitative records')
N_SHOW = len(recs)
fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1: axes = axes[np.newaxis, :]
fig.suptitle(f'Attention U-Net + Prompt Crop - {N_SHOW} shared test images (center_shift, same stems as center_zoom)', fontsize=13, y=1.001)
for ax, title in zip(axes[0], ['Input image', 'Prompt box', 'Prediction', 'Ground truth', 'TP/FP/FN']):
    ax.set_title(title, fontsize=10, fontweight='bold')
for row_idx, rec in enumerate(recs):
    img_np = rec['img']; pred_np = rec['pred']; gt_np = rec['gt']
    dice, iou, precision, recall = dice_iou_pre_rec(pred_np, gt_np)
    row = axes[row_idx]; bg = np.stack([img_np] * 3, axis=-1)
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{row_idx+1} [{rec["n_samples"]}p]\nDice={dice:.3f}', fontsize=8)
    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    for bx0, by0, bx1, by1 in rec['bboxes']:
        row[1].add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0,
                                   edgecolor='cyan', fill=False, linewidth=1.5))
    row[2].imshow(bg); row[2].imshow(np.where(pred_np > 0, pred_np, np.nan),
                                    cmap='Blues', alpha=0.60, vmin=0, vmax=1)
    row[3].imshow(bg); row[3].imshow(np.where(gt_np > 0, gt_np, np.nan),
                                    cmap='Greens', alpha=0.60, vmin=0, vmax=1)
    overlay = np.zeros((*gt_np.shape, 3), dtype=float)
    overlay[..., 1] = pred_np * gt_np
    overlay[..., 0] = pred_np * (1 - gt_np)
    overlay[..., 2] = (1 - pred_np) * gt_np
    row[4].imshow(bg); row[4].imshow(overlay, alpha=0.75 * np.any(overlay > 0, axis=-1), vmin=0, vmax=1)
    row[4].text(0.02, 0.02, f'Dice={dice:.3f}\nIoU={iou:.3f}\nPrecision={precision:.3f}\nRecall={recall:.3f}',
                transform=row[4].transAxes, fontsize=8, va='bottom', ha='left', color='white',
                bbox=dict(facecolor='black', alpha=0.45, pad=3, edgecolor='none'))
    for ax in row:
        ax.axis('off')
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False, edgecolor='#555555', linewidth=1.0, clip_on=False))
fig.legend(handles=[Patch(facecolor='green', label='TP'), Patch(facecolor='red', label='FP'),
                    Patch(facecolor='blue', label='FN')],
           loc='lower center', ncol=3, frameon=False)
plt.tight_layout(); plt.subplots_adjust(bottom=0.06)
export_qualitative_rows(fig, axes, recs)


---
## Summary & Comparison
---

In [ ]:
# ══════════════════════════════════════════════════════
# SUMMARY - PGA-UNet vs Attention U-Net vs Prompt-Matched Conventional Baselines (IMG_SIZE=512)
# Load from CSV if the variables are not already in memory
# ══════════════════════════════════════════════════════
import csv, os, numpy as np
import matplotlib.pyplot as plt

BASE      = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
PGA_ROOT  = f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'
UNET_ROOT = f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'
KEYS      = ['dice','iou','precision','recall','hd95','cbl']

# ── Load results: prioritize in-memory variables, otherwise fall back to CSV ──────────────
if 'pga_results' not in dir() or not pga_results:
    with open(f'{PGA_ROOT}/results/pga_unet2d_test_results.csv') as _f:
        pga_results = {}
        for _row in csv.DictReader(_f):
            pga_results[_row['mode']] = {k: float(_row[k]) for k in KEYS}
    print('Loaded `pga_results` from CSV')
else:
    print('Using in-memory `pga_results`')

if 'att_unet_results' not in dir() or not att_unet_results:
    with open(f'{UNET_ROOT}/results/attention_att_unet_results.csv') as _f:
        _row = next(csv.DictReader(_f))
        att_unet_results = {k: float(_row[k]) for k in KEYS}
    print('Loaded `att_unet_results` from CSV')
else:
    print('Using in-memory `att_unet_results`')

if 'concat_results' not in dir() or not concat_results:
    concat_results = {}
    with open(f'{PGA_ROOT}/results/attunet_concat_prompt_results.csv') as _f:
        for _row in csv.DictReader(_f):
            concat_results[_row['mode']] = {k: float(_row[k]) for k in KEYS}
    print('Loaded `concat_results` from CSV')
else:
    print('Using in-memory `concat_results`')

# crop_results uses different key names (dice_full/iou_full/... instead of
# dice/iou/...) because it also reports a separate dice_crop diagnostic;
# remap to the shared KEYS naming here for a uniform summary table.
_CROP_KEY_MAP = {'dice': 'dice_full', 'iou': 'iou_full', 'precision': 'pre_full',
                  'recall': 'rec_full', 'hd95': 'hd95_full', 'cbl': 'cbl_full'}
if 'crop_results' not in dir() or not crop_results:
    crop_results_raw = {}
    with open(f'{PGA_ROOT}/results/attunet_crop_prompt_results.csv') as _f:
        for _row in csv.DictReader(_f):
            crop_results_raw[_row['scenario']] = {
                k: float(_row[v]) for k, v in _CROP_KEY_MAP.items()
            }
    crop_results_summary = crop_results_raw
    print('Loaded `crop_results` from CSV')
else:
    crop_results_summary = {
        mode: {k: crop_results[mode][v] for k, v in _CROP_KEY_MAP.items()}
        for mode in crop_results
    }
    print('Using in-memory `crop_results`')

HDRS   = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓ (px)','CBL↑']
pga_zo = pga_results['center_zoom']
pga_sh = pga_results['center_shift']
concat_zo = concat_results['center_zoom']
concat_sh = concat_results['center_shift']
crop_zo = crop_results_summary['center_zoom']
crop_sh = crop_results_summary['center_shift']

# ── Summary table ────────────────────────────────────────────────────
ROWS = [
    ('PGA-UNet (center_zoom)',                    pga_zo),
    ('PGA-UNet (center_shift)',                   pga_sh),
    ('Attention U-Net',                           att_unet_results),
    ('AttUNet + Prompt Channel (center_zoom)',    concat_zo),
    ('AttUNet + Prompt Channel (center_shift)',   concat_sh),
    ('AttUNet + Prompt Crop (center_zoom)',       crop_zo),
    ('AttUNet + Prompt Crop (center_shift)',      crop_sh),
]

bar='═'*98
print(f'\n{bar}')
print('  COMPARISON - PGA-UNet vs Attention U-Net vs Prompt-Matched Conventional Baselines (image-level, IMG_SIZE=512)')
print(f'{bar}')
print(f"  {'Model':<40}"+''.join(f'{h:>9}' for h in HDRS))
print('  '+'-'*95)
for name, r in ROWS:
    print(f"  {name:<40}"+''.join(f'{r[k]:>9.4f}' for k in KEYS))
print(f'{bar}')
for name, r in ROWS[1:]:
    print(f"  Delta [PGA-UNet (center_zoom)] - [{name}]:")
    for k in KEYS:
        delta = pga_zo[k] - r[k]
        suffix = '  (negative means PGA is better)' if k == 'hd95' else ''
        print(f'    {k.upper():<10}: {delta:+.4f}{suffix}')

# ── Bar charts: one subplot per metric, one bar per row above ──────────
METRIC_PAIRS = [
    ('dice',      'Dice ↑'),
    ('iou',       'IoU ↑'),
    ('precision', 'Precision ↑'),
    ('recall',    'Recall ↑'),
    ('hd95',      'HD95 ↓ (px)'),
    ('cbl',       'CBL ↑'),
]

ROW_COLORS = ['#1565C0', '#1976D2', '#EF5350', '#43A047', '#66BB6A', '#FB8C00', '#FFA726']
ROW_LABELS = ['PGA-UNet\n(center_zoom)', 'PGA-UNet\n(center_shift)', 'Attention U-Net\n(no prompt)',
              'AttUNet+Channel\n(center_zoom)', 'AttUNet+Channel\n(center_shift)',
              'AttUNet+Crop\n(center_zoom)', 'AttUNet+Crop\n(center_shift)']
x_pos = list(range(len(ROWS)))

fig, axes = plt.subplots(2, 3, figsize=(22, 10))
axes = axes.flatten()
fig.suptitle('PGA-UNet vs Attention U-Net vs prompt-matched conventional baselines - 6 metrics (IMG_SIZE=512)',
             fontsize=14, fontweight='bold', y=1.02)

for ax_idx, (metric, label) in enumerate(METRIC_PAIRS):
    ax = axes[ax_idx]
    vals = [r[metric] for _, r in ROWS]

    ax.bar(x_pos, vals, 0.6, color=ROW_COLORS, alpha=0.88, edgecolor='white')
    for xi, val in zip(x_pos, vals):
        fmt = f'{val:.0f}' if metric == 'hd95' else f'{val:.3f}'
        ax.text(xi, val + (max(vals) * 0.015 if metric != 'hd95' else max(vals) * 0.02), fmt,
                ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_title(label, fontsize=12, fontweight='bold', pad=8)
    hi = max(vals)
    ax.set_ylim(0, hi * 1.20 if metric != 'hd95' else hi * 1.25)
    ax.set_xticks(x_pos); ax.set_xticklabels(ROW_LABELS, fontsize=7.5, rotation=20, ha='right')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{BASE}/pga_vs_attention_unet_and_baselines_comparison.png', dpi=150, bbox_inches='tight')
from IPython.display import display as _d; _d(fig); plt.close(fig)
print('\n✅ Figure saved: pga_vs_attention_unet_and_baselines_comparison.png')

---
## Small-lesion R512 architecture comparison


In [ ]:
# SMALL-LESION R512 ARCHITECTURE ANALYSIS
# The 50 images are selected only by total GT-union area in the shared 512 frame.
from collections import OrderedDict
from dataset import PromptSegmentationDataset
small_source=all_image_records['center_zoom']
small_names=[r['img_name'] for r in sorted(small_source,key=lambda r:float((r['gt']>0.5).sum()))[:50]]
small_set=set(small_names)
print(f'Small-lesion subset N={len(small_names)}, defined independently of model predictions')

def avg_records(records):
    return {k:float(np.mean([r[k] for r in records])) for k in KEYS}
def take(records): return [r for r in records if r['img_name'] in small_set]

SMALL={}
SMALL['att_unet']=avg_records([r for r in att_image_records if r['img_name'] in small_set])
for mode in ['center_zoom','center_shift']:
    SMALL[f'pga512_{mode}']=avg_records(take(all_image_records[mode]))
    SMALL[f'channel512_{mode}']=avg_records(take(concat_image_records[mode]))
    cr=[r for r in crop_raw_results[mode]['img_recs'] if r['img_name'] in small_set]
    SMALL[f'crop512_{mode}']={
        'dice':float(np.mean([r['dice_full'] for r in cr])), 'iou':float(np.mean([r['iou_full'] for r in cr])),
        'precision':float(np.mean([r['pre_full'] for r in cr])), 'recall':float(np.mean([r['rec_full'] for r in cr])),
        'hd95':float(np.mean([r['hd95_full'] for r in cr])), 'cbl':float(np.mean([r['cbl_full'] for r in cr]))}


rows=[('Attention U-Net 512','no_prompt','att_unet')]
for mode in ['center_zoom','center_shift']:
    rows += [('AttUNet + Prompt Channel 512',mode,f'channel512_{mode}'),('AttUNet + Prompt Crop 512',mode,f'crop512_{mode}'),('PGA-UNet 512',mode,f'pga512_{mode}')]
print('\n'+'='*112); print('SMALL-LESION R512 ARCHITECTURE COMPARISON'); print('='*112)
csv_rows=[]
for model,mode,key in rows:
    m=SMALL[key]; print(f"{model:<34} {mode:<13}"+''.join(f'{m[k]:>10.4f}' for k in KEYS)); csv_rows.append([model,mode,50]+[f'{m[k]:.4f}' for k in KEYS])

os.makedirs('results',exist_ok=True)
with open('results/subcat_small_r512.csv','w',newline='') as f: csv.writer(f).writerows([['model','scenario','N']+KEYS]+csv_rows)
